# Week 10 Notebook: VoltCore Energy — Plant Closure (Part 5)

*OMAIB · Module 1 · Week 10*

*© H. Sami Karaca. Questrom School of Business, Boston University.
Online Master of Science in Artificial Intelligence in Business.
Licensed for use in this course only; not for redistribution.*

**What this notebook is.** This is a practice notebook: it is not submitted and does not affect your grade. It runs the mixed-integer program behind the closure recommendation you met on the case page, stress-tests the cost assumptions driving it, and produces every number your board memo needs.

**How to read every output.** Solution status, variable values, objective value — and, new this week, the **optimality gap**: the solver's provable bound on how far its answer can be from the true best. With four plants there are only sixteen open/close configurations, so expect the gap to be exactly zero.

## The Case (Part 5)

*The block below — together with the Part 4 network tables that follow — is the full case description. This is what you copy into your GenAI tool for Step 1 on the Colab page, before running any cells.*

---

A private equity firm is evaluating VoltCore. The network earns roughly \$1,936 million a year in gross contribution — and carries large annual fixed costs, the price of simply keeping each plant open, regardless of how much it produces:

| Plant | Annual fixed cost (\$M) | Note |
| --- | --- | --- |
| Georgia (Coweta County) | 300 | built with a state incentive package |
| Nevada | 458 | built with a state incentive package |
| Gdansk | 350 | |
| Ulsan | 325 | |

At present all four plants run, the network operates at about 78% of capacity, and net profit is roughly \$503 million per year. The PE thesis is blunt: the network has more capacity than the market needs, so closing one or more plants would eliminate their fixed costs and concentrate production in the most efficient facilities.

Closing a plant eliminates its fixed cost entirely and removes all of its capacity; keeping it open incurs the full fixed cost and makes its capacity available. VoltCore is still not required to serve all demand. **Decide which plants stay open and how much each open plant produces, to maximize annual net profit (contribution minus fixed costs).**

The production, market, and lane data are unchanged from Part 4:

| Plant | Annual capacity (GWh) | Production cost (\$/kWh) |
| --- | --- | --- |
| Georgia | 10 | 58.17 |
| Nevada | 20 | 53.67 |
| Gdansk | 12 | 63.92 |
| Ulsan | 25 | 39.00 |

| Market | Max demand (GWh) | Selling price (\$/kWh) |
| --- | --- | --- |
| Atlanta | 15 | 92.50 |
| Sao Paulo | 7.5 | 101.83 |
| Rotterdam | 20 | 96.33 |
| Shanghai | 10 | 104.42 |

Transport + tariff cost (\$/kWh), by plant (row) and market (column):

| From \\ To | Atlanta | Sao Paulo | Rotterdam | Shanghai |
| --- | --- | --- | --- | --- |
| Georgia | 0.00 | 20.38 | 15.13 | 35.75 |
| Nevada | 7.50 | 27.50 | 22.25 | 29.75 |
| Gdansk | 15.25 | 21.00 | 0.00 | 20.88 |
| Ulsan | 35.75 | 30.75 | 25.25 | 9.88 |

# Environment Setup

Before running any cells, select **File › Save a copy in Drive** so your work saves to your own Google Drive. This notebook loads everything it needs automatically — there's nothing for you to upload. Run the cells in order. This version uses the libraries already in your session. Results may vary with the runtime; use your own output in your answers.

In [ ]:
# Install the Pyomo optimization modeling package (+ tabulate for output tables)
!pip install -q pyomo tabulate

# Install the optimization engine (solution algorithms)
# GLPK solves linear AND mixed-integer programs: https://www.gnu.org/software/glpk/
!apt-get install -y -qq glpk-utils

import pyomo.environ as pyo
from pyomo.opt import SolverFactory
from tabulate import tabulate as tb

## The Model

Same sixteen lane variables as Part 4 (`x11` = Georgia to Atlanta ... `x44` = Ulsan to Shanghai), plus what makes this a **mixed-integer program**: four binary variables, `Y1` through `Y4`, one per plant — 1 means open, 0 means closed. Each plant's capacity constraint is multiplied by its binary, so a closed plant can ship nothing, and its fixed cost is charged only if its binary is 1.

The function below builds and solves the whole model. Its arguments exist for the sensitivity work later — the base run uses the defaults.

In [ ]:
def solve_closure(georgia_cost_mult=1.0, gdansk_cost_mult=1.0, force_open=(), min_open=0,
                  serve_sp_full=False):
    """Build and solve the VoltCore Part 5 closure MIP. Defaults are the base case.
    georgia_cost_mult / gdansk_cost_mult scale those plants' production costs
    (0.94 = a 6% cost reduction). force_open lists plants that must stay open.
    min_open sets a minimum number of open plants. serve_sp_full=True adds the
    tie-break constraint explained below the base run: Sao Paulo fully served."""

    model = pyo.ConcreteModel()

    # ------------------------------------
    # Decision Variables                  |
    # ------------------------------------
    # Lane volumes (GWh), first index = plant, second = market
    for name in ['x11','x12','x13','x14','x21','x22','x23','x24',
                 'x31','x32','x33','x34','x41','x42','x43','x44']:
        setattr(model, name, pyo.Var(bounds=(0, None)))
    x11, x12, x13, x14 = model.x11, model.x12, model.x13, model.x14   # Georgia ->
    x21, x22, x23, x24 = model.x21, model.x22, model.x23, model.x24   # Nevada ->
    x31, x32, x33, x34 = model.x31, model.x32, model.x33, model.x34   # Gdansk ->
    x41, x42, x43, x44 = model.x41, model.x42, model.x43, model.x44   # Ulsan ->

    # NEW: one binary per plant -- 1 = keep open, 0 = close
    model.Y1 = pyo.Var(within=pyo.Boolean)   # Georgia
    model.Y2 = pyo.Var(within=pyo.Boolean)   # Nevada
    model.Y3 = pyo.Var(within=pyo.Boolean)   # Gdansk
    model.Y4 = pyo.Var(within=pyo.Boolean)   # Ulsan
    Y1, Y2, Y3, Y4 = model.Y1, model.Y2, model.Y3, model.Y4

    georgia_cost = 58.17 * georgia_cost_mult
    gdansk_cost  = 63.92 * gdansk_cost_mult

    # ------------------------------------
    # Objective Function ($M): net profit |
    # ------------------------------------
    # contribution (as in Part 4) minus the fixed cost of every open plant
    model.obj = pyo.Objective(expr =
          92.50*(x11 + x21 + x31 + x41)        # Atlanta selling price
        + 101.83*(x12 + x22 + x32 + x42)       # Sao Paulo selling price
        +  96.33*(x13 + x23 + x33 + x43)       # Rotterdam selling price
        + 104.42*(x14 + x24 + x34 + x44)       # Shanghai selling price
        - georgia_cost*(x11 + x12 + x13 + x14) # Georgia production cost
        -  53.67*(x21 + x22 + x23 + x24)       # Nevada production cost
        - gdansk_cost*(x31 + x32 + x33 + x34)  # Gdansk production cost
        -  39.00*(x41 + x42 + x43 + x44)       # Ulsan production cost
        - (  0.00*x11 + 20.38*x12 + 15.13*x13 + 35.75*x14    # Georgia lanes
           + 7.50*x21 + 27.50*x22 + 22.25*x23 + 29.75*x24    # Nevada lanes
           +15.25*x31 + 21.00*x32 +  0.00*x33 + 20.88*x34    # Gdansk lanes
           +35.75*x41 + 30.75*x42 + 25.25*x43 +  9.88*x44)   # Ulsan lanes
        - (300*Y1 + 458*Y2 + 350*Y3 + 325*Y4)                # fixed costs of open plants
        , sense = pyo.maximize)

    # ------------------------------------
    # Constraints                         |
    # ------------------------------------
    # Capacity is available only if the plant is open (binary = 1)
    model.CapGeorgia = pyo.Constraint(expr = x11 + x12 + x13 + x14 <= 10*Y1)
    model.CapNevada  = pyo.Constraint(expr = x21 + x22 + x23 + x24 <= 20*Y2)
    model.CapGdansk  = pyo.Constraint(expr = x31 + x32 + x33 + x34 <= 12*Y3)
    model.CapUlsan   = pyo.Constraint(expr = x41 + x42 + x43 + x44 <= 25*Y4)

    # Market demand ceilings -- ceilings, not obligations
    model.DemAtlanta   = pyo.Constraint(expr = x11 + x21 + x31 + x41 <= 15)
    model.DemSaoPaulo  = pyo.Constraint(expr = x12 + x22 + x32 + x42 <= 7.5)
    model.DemRotterdam = pyo.Constraint(expr = x13 + x23 + x33 + x43 <= 20)
    model.DemShanghai  = pyo.Constraint(expr = x14 + x24 + x34 + x44 <= 10)

    # Optional policy constraints (used in the price-the-constraint step)
    Ymap = {'Georgia': Y1, 'Nevada': Y2, 'Gdansk': Y3, 'Ulsan': Y4}
    model.policies = pyo.ConstraintList()
    for plant in force_open:
        model.policies.add(Ymap[plant] >= 1)
    if min_open:
        model.policies.add(Y1 + Y2 + Y3 + Y4 >= min_open)
    if serve_sp_full:
        # Tie-break: keep Sao Paulo's 7.5 GWh fully served (see the note below the base run)
        model.policies.add(x12 + x22 + x32 + x42 >= 7.5)

    # ------------------------------------
    # Solve the Model                     |
    # ------------------------------------
    opt = SolverFactory('glpk')
    results = opt.solve(model, symbolic_solver_labels=True)
    return model, results


def closure_report(m, results):
    """Print status, open/close decisions, net profit, and the optimality gap."""
    print('Solution status :', results.solver.termination_condition)
    try:
        lb = results.problem.lower_bound
        ub = results.problem.upper_bound
        gap = abs(ub - lb)
        print('Optimality gap  :', f'{gap:.4f}', '(proven optimal)' if gap < 1e-6 else '')
    except Exception:
        print('Optimality gap  : not reported by solver')
    print()
    flags = [['Georgia', round(m.Y1())], ['Nevada', round(m.Y2())],
             ['Gdansk', round(m.Y3())], ['Ulsan', round(m.Y4())]]
    print(tb([[p, 'OPEN' if y else 'CLOSED'] for p, y in flags],
             headers=['Plant', 'Decision'], tablefmt='fancy_grid'))
    print()
    print('Annual net profit :', f'${m.obj():,.1f}M')

## The Status Quo — All Four Plants Open

First, the baseline the PE firm wants to beat: force every binary to 1 and let the model allocate production. This is the network as it runs today.

In [ ]:
status_quo, sq_results = solve_closure(force_open=('Georgia', 'Nevada', 'Gdansk', 'Ulsan'))

print('--------------- STATUS QUO: ALL FOUR PLANTS OPEN --------------- \n')
closure_report(status_quo, sq_results)

## The Acquisition Model — Free to Close

You committed to your own closure plan on the case page. Now remove the force-open handcuffs and let the model choose. Read the output top to bottom: status, gap, decisions, net profit.

In [ ]:
base, results = solve_closure()

print('--------------- THE CLOSURE DECISION --------------- \n')
closure_report(base, results)
print('Uplift vs status quo :', f'${base.obj() - status_quo.obj():,.1f}M',
      f'({(base.obj() - status_quo.obj()) / status_quo.obj() * 100:.0f}%)')

### One more judgment call — breaking a tie

Before reading the shipping plan, one subtlety. Ulsan earns *exactly* the same margin in Sao Paulo (\$101.83 − \$39.00 − \$30.75 = \$32.08 per kWh) as in Rotterdam (\$96.33 − \$39.00 − \$25.25 = \$32.08). So two different shipping plans earn exactly the same \$826.9M: the solver can put the whole 7.5 GWh shortfall in Rotterdam, or split it between Rotterdam and Sao Paulo. **The model is indifferent — management is not.** VoltCore chooses to keep Sao Paulo, its smallest market, fully served rather than spread the shortfall across two customer relationships. We encode that judgment as one constraint (`serve_sp_full=True` adds *Sao Paulo shipments ≥ 7.5*), re-solve, and check the price of the choice: if the profit is unchanged, the constraint only chose between ties — it cost nothing.


In [ ]:
# The model is indifferent between two shipping plans -- management is not.
# serve_sp_full=True adds one constraint: Sao Paulo's 7.5 GWh stays fully served.
plan, plan_results = solve_closure(serve_sp_full=True)

print('Net profit, solver left to choose  :', f'${base.obj():,.2f}M')
print('Net profit, Sao Paulo kept whole   :', f'${plan.obj():,.2f}M')
print()
print('Identical to the penny: the constraint chose between ties -- it cost nothing.')


In [ ]:
# Where does the surviving network ship -- and which demand goes unserved?
def shipment_tables(m):
    g = lambda v: 0.0 if abs(v) < 1e-9 else round(v, 2)
    plan = [['From \\ To', 'Atlanta', 'Sao Paulo', 'Rotterdam', 'Shanghai'],
            ['Georgia', g(m.x11()), g(m.x12()), g(m.x13()), g(m.x14())],
            ['Nevada',  g(m.x21()), g(m.x22()), g(m.x23()), g(m.x24())],
            ['Gdansk',  g(m.x31()), g(m.x32()), g(m.x33()), g(m.x34())],
            ['Ulsan',   g(m.x41()), g(m.x42()), g(m.x43()), g(m.x44())]]
    print(tb(plan, headers='firstrow', tablefmt='fancy_grid'))

    caps = [['Georgia', m.x11()+m.x12()+m.x13()+m.x14(), 10],
            ['Nevada',  m.x21()+m.x22()+m.x23()+m.x24(), 20],
            ['Gdansk',  m.x31()+m.x32()+m.x33()+m.x34(), 12],
            ['Ulsan',   m.x41()+m.x42()+m.x43()+m.x44(), 25]]
    print()
    print(tb([[p, f'{g(u):,.1f}', f'{c:,.1f}'] for p, u, c in caps],
             headers=['Plant', 'GWh shipped', 'Capacity (GWh)'], tablefmt='fancy_grid'))
    # Closed plants ship zero -- their capacity left with them. The survivors run flat out:
    # the 78%-utilization network became a 100%-utilization one.

    dems = [['Atlanta',   m.x11()+m.x21()+m.x31()+m.x41(), 15],
            ['Sao Paulo', m.x12()+m.x22()+m.x32()+m.x42(), 7.5],
            ['Rotterdam', m.x13()+m.x23()+m.x33()+m.x43(), 20],
            ['Shanghai',  m.x14()+m.x24()+m.x34()+m.x44(), 10]]
    unserved = sum(c - u for _, u, c in dems)
    print()
    print(tb([[mk, f'{g(u):,.1f}', f'{c:,.1f}'] for mk, u, c in dems],
             headers=['Market', 'GWh received', 'Demand ceiling (GWh)'], tablefmt='fancy_grid'))
    print()
    print('Total demand left unserved :', f'{unserved:,.1f} GWh')

shipment_tables(plan)

**Check your reading.** The model closes **Georgia and Gdansk** and keeps Nevada and Ulsan, lifting annual net profit from roughly **\$503M** to roughly **\$827M** — an uplift of about **\$324M, or 64%** — with the optimality gap at zero: sixteen configurations, every one checked, proven best.

Notice the quietest number in the tables: **7.5 GWh of demand goes unserved** — all of it Rotterdam's. The surviving network delivers 12.5 of Rotterdam's 20 GWh, abandoning **37%** of that market, exactly as the case pages describe. Without Gdansk's zero-cost European lane, that demand is no longer worth serving. And remember which part of this plan was *whose* decision: the closures, the profit, and the 7.5 GWh shortfall came from the optimizer; *where* the shortfall lands came from the tie-break constraint — a management judgment the model could not make on its own. Constraints are not just physics; they are where values enter the model. Keep that thought — the Ethics Deliberation builds on it.

If you asked your GenAI tool for a recommendation (Step 1 on the Colab page), set its configuration and numbers beside these — that comparison is your final audit entry. One caution as you compare: if a tool reports the same closures and the same \$826.9M profit but a different shipping split, it has not made an error — it found the other end of the tie. Compare closures and profit first; those are the facts. The shipping split is a judgment call the model can't make for you.


## Sensitivity Sweep — How Fragile Is the Closure Call?

The board will ask: what would have to be different for this recommendation to change? The sweep below answers with a number. It re-solves the model at a grid of production-cost reductions for one plant and reports where the closure decision flips.

In [ ]:
# VoltCore closure -- sensitivity analysis range (modify this)
SENSITIVITY_RANGE = (-0.10, 0)      # sweep production-cost reductions of up to 10%

def sweep_plant(plant, step=0.005):
    """Re-solve at each cost reduction and report when the plant's closure flips."""
    print(f'--------------- SENSITIVITY SWEEP: {plant.upper()} PRODUCTION COST --------------- \n')
    rows, last_closed, flipped = [], None, None
    r = 0.0
    while r >= SENSITIVITY_RANGE[0] - 1e-9:   # walk cost reductions across the range
        mult = 1.0 + r
        kwargs = {'georgia_cost_mult': mult} if plant == 'Georgia' else {'gdansk_cost_mult': mult}
        m, _ = solve_closure(**kwargs)
        is_open = round({'Georgia': m.Y1, 'Gdansk': m.Y3}[plant]())
        rows.append([f'{-r*100:.1f}%', 'OPEN' if is_open else 'closed', f'${m.obj():,.1f}M'])
        if is_open and flipped is None:
            flipped = -r
        if not is_open and flipped is None:
            last_closed = -r
        r -= step
    print(tb(rows, headers=['Cost reduction', f'{plant} decision', 'Net profit'], tablefmt='fancy_grid'))
    if flipped is not None:
        # A grid search can only bracket the flip: it lies somewhere between the last
        # 'closed' grid point and the first 'OPEN' one. Shrink `step` to tighten it.
        print(f'\nThe closure recommendation for {plant} flips between a '
              f'{last_closed*100:.1f}% and a {flipped*100:.1f}% cost reduction.')

sweep_plant('Georgia')

In [ ]:
sweep_plant('Gdansk')

**Check your reading.** Georgia's closure flips between a **5.5% and 6% cost reduction**; Gdansk's between **4% and 4.5%** — Gdansk's recommendation is the more fragile of the two. The sweep walks a half-percent grid, so it brackets each flip rather than pinpointing it; the exact breakevens sit inside those brackets, at roughly **5.7%** for Georgia and **4.3%** for Gdansk — the "roughly 6%" and "roughly 4%" you will meet again in the Ethics Deliberation. (Want a tighter bracket? Shrink `step` and rerun.) These are narrow margins for a decision with this much downstream exposure, and they are exactly the numbers your board memo's final question should put against current procurement projections.

Also notice *what happens* when either plant earns its way back in: the returning plant does not simply join the club — the optimal configuration reshuffles. Run the sweep tables again and watch the net-profit column.


## Optional — Price the Constraint Yourself

The board asks: "What would it cost us to protect the Georgia plant and its workforce?" That question has a dollar answer. Force Georgia's binary to 1 and rerun — and look carefully at *which plants* the new plan keeps.

In [ ]:
protected, p_results = solve_closure(force_open=('Georgia',))

print('--------------- POLICY: GEORGIA STAYS OPEN --------------- \n')
closure_report(protected, p_results)
print('Price of the constraint :', f'${base.obj() - protected.obj():,.1f}M per year')

# Try it: the well-meant policy "keep at least three plants open" --
# replace the line above with solve_closure(min_open=3) and notice who closes.

**Check your reading.** Protecting Georgia costs about **\$33M a year** — roughly 4% of the optimized profit. But look at the decisions table: the model satisfies the constraint by keeping Georgia *and Gdansk*, and closing **Nevada** — the plant the unconstrained optimum kept. A constraint doesn't just cost money; it reshapes the whole solution, and it can choose who pays. (The "keep at least three plants open" policy in the commented line lands in the same place: Nevada closes.) That number — the price of the constraint — is the live session's central takeaway, and now you can set it yourself.

---
## Before You Leave

Your board memo needs: what the model optimized (net profit — contribution minus fixed costs), what it recommends (\$503M to \$827M), what it never priced, and the flip thresholds from the sweeps. Paste the memo into the **VoltCore Audit Log** assignment on Blackboard along with your final audit entry — the four comparison lenses and the three-week synthesis question. The notebook itself is not submitted.

*This closes the three-week arc: a linear program proved its plan, a network outran your intuition, and a mixed-integer program just recommended closing two plants — provably, optimally, and in total silence about everything it was never asked to count. What you do with that silence is the week's real subject.*